# Sim 5c: PDA vs. CoT Distillation

**Question:** Is multi-perspective reasoning (PDA) better than single-perspective
chain-of-thought (CoT) for distillation?

**Design:** Train two LoRA adapters on the same questions, same model, same
hyperparameters. Only difference: PDA training data (3 workers + merge) vs.
CoT training data (single pass "think step by step").

**Upload these files:**
- PDA data: `pda_training_data.jsonl`, `pda_math_training.jsonl`, `pda_arc_training.jsonl`
- CoT data: `cot_gsm8k_training.jsonl`, `cot_math_training.jsonl`, `cot_arc_training.jsonl`
- Model (if HF is down): `Qwen3-1.7B/` folder in Google Drive

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets

## 1. Load Training Data

In [ ]:
import json, re, random

def load_correct(path, domain, reasoning_key="pda_reasoning"):
    examples = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            if d.get("correct", False):
                examples.append({
                    "question": d["question"],
                    "reasoning": d[reasoning_key],
                    "domain": domain,
                })
    return examples

# PDA data
pda_data = (
    load_correct("pda_training_data.jsonl", "gsm8k", "pda_reasoning") +
    load_correct("pda_math_training.jsonl", "math", "pda_reasoning") +
    load_correct("pda_arc_training.jsonl", "arc", "pda_reasoning")
)

# CoT data
cot_data = (
    load_correct("cot_gsm8k_training.jsonl", "gsm8k", "cot_reasoning") +
    load_correct("cot_math_training.jsonl", "math", "cot_reasoning") +
    load_correct("cot_arc_training.jsonl", "arc", "cot_reasoning")
)

random.seed(42)
random.shuffle(pda_data)
random.shuffle(cot_data)

print(f"PDA: {len(pda_data)} examples")
print(f"CoT: {len(cot_data)} examples")

## 2. Shared Setup

In [ ]:
import os, torch
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

max_seq_length = 2048

# Try local Drive path first, fall back to HF download
MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/models/Qwen3-1.7B"
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = "Qwen/Qwen3-1.7B"  # will download from HF
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("TRANSFORMERS_OFFLINE", None)

print(f"Model path: {MODEL_PATH}")

SYSTEM_PROMPT = """You are a problem solver who considers multiple perspectives.
1. Solve systematically, showing clear steps.
2. Look for more efficient approaches.
3. Check for edge cases and common mistakes.
4. Synthesize the best answer."""

def make_dataset(data, tokenizer):
    formatted = [{"conversations": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["reasoning"]},
    ]} for ex in data]
    ds = Dataset.from_list(formatted)
    return ds.map(lambda examples: {"text": [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples["conversations"]
    ]}, batched=True)

def train_adapter(model, tokenizer, dataset, output_dir):
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=dataset, dataset_text_field="text",
        max_seq_length=max_seq_length, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=10, num_train_epochs=3, learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10, optim="adamw_8bit",
            weight_decay=0.01, lr_scheduler_type="linear",
            seed=42, output_dir=output_dir,
        ),
    )
    stats = trainer.train()
    print(f"Training loss: {stats.training_loss:.4f}")
    model.save_pretrained(output_dir)
    return stats

## 3. Train PDA Adapter

In [ ]:
# Mount Drive for local model
from google.colab import drive
drive.mount('/content/drive')

# Load fresh model for PDA adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=max_seq_length, dtype=None, load_in_4bit=False,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print(f"\n=== Training PDA adapter on {len(pda_data)} examples ===")
pda_dataset = make_dataset(pda_data, tokenizer)
train_adapter(model, tokenizer, pda_dataset, "adapter-pda")
print("PDA adapter saved.")

## 4. Train CoT Adapter

In [ ]:
# Reload fresh model for CoT adapter (clean weights)
del model
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=max_seq_length, dtype=None, load_in_4bit=False,
)
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

print(f"\n=== Training CoT adapter on {len(cot_data)} examples ===")
cot_dataset = make_dataset(cot_data, tokenizer)
train_adapter(model, tokenizer, cot_dataset, "adapter-cot")
print("CoT adapter saved.")

## 5. Evaluation — Base vs PDA vs CoT

In [ ]:
from datasets import load_dataset
from peft import PeftModel

N_EVAL = 200
random.seed(42)

# --- Helpers ---
def extract_number(text):
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    if m: return float(m.group(1).replace(",", ""))
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    for n in reversed(nums):
        c = n.replace(",", "").strip()
        if c and c != "-":
            try: return float(c)
            except: continue
    return None

def extract_boxed(text):
    m = re.search(r'\\boxed\{([^}]+)\}', text)
    return m.group(1).strip() if m else None

def extract_mc(text):
    m = re.search(r'(?:answer is|Answer:?)\s*\(?([A-E])\)?\.?', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r'\(?([A-E])\)\s*$', text.strip())
    return m.group(1).upper() if m else None

def normalize(s):
    if s is None: return None
    s = str(s).strip().replace(" ", "").lower()
    if s.endswith("."): s = s[:-1]
    try: return str(float(s))
    except: return s

def generate(model, tokenizer, question):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        out = model.generate(input_ids=ids, max_new_tokens=256,
                            temperature=0.0, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

# --- Load test sets ---
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

print("Loading test sets...")
gsm8k_test = load_dataset("openai/gsm8k", "main", split="test")
math_test = load_dataset("EleutherAI/hendrycks_math", "algebra", split="test")
arc_test = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

def sample(ds, n):
    idx = list(range(len(ds)))
    random.shuffle(idx)
    return idx[:n]

gsm8k_idx = sample(gsm8k_test, N_EVAL)
math_idx = sample(math_test, N_EVAL)
arc_idx = sample(arc_test, N_EVAL)

def gt_gsm(i):
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', gsm8k_test[i]["answer"])
    return float(m.group(1).replace(",", "")) if m else None
def gt_math(i): return extract_boxed(math_test[i]["solution"])
def gt_arc(i): return arc_test[i]["answerKey"]
def fmt_gsm(i): return gsm8k_test[i]["question"]
def fmt_math(i): return math_test[i]["problem"]
def fmt_arc(i):
    item = arc_test[i]
    q = item["question"]
    for l, t in zip(item["choices"]["label"], item["choices"]["text"]):
        q += f"\n({l}) {t}"
    return q

benches = [
    ("GSM8K", gsm8k_idx, fmt_gsm, gt_gsm, extract_number),
    ("MATH",  math_idx,  fmt_math, gt_math, extract_boxed),
    ("ARC-C", arc_idx,   fmt_arc,  gt_arc,  extract_mc),
]

def eval_all(model, tokenizer, tag):
    results = {}
    for name, idx, fmt, gt, ext in benches:
        correct = total = 0
        for i, ix in enumerate(idx):
            g = gt(ix)
            if g is None: continue
            resp = generate(model, tokenizer, fmt(ix))
            pred = ext(resp)
            if pred is not None and normalize(str(pred)) == normalize(str(g)):
                correct += 1
            total += 1
            if (i+1) % 50 == 0:
                print(f"  [{tag}] {name}: {i+1}/{len(idx)} -- {correct}/{total} ({100*correct/total:.1f}%)")
        acc = round(100*correct/total, 1) if total else 0
        results[name] = {"correct": correct, "total": total, "accuracy": acc}
        print(f"  [{tag}] {name}: {correct}/{total} ({acc}%)")
    return results

print("Ready.")

In [ ]:
# Load fresh model for evaluation
del model
torch.cuda.empty_cache()

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=max_seq_length, dtype=None, load_in_4bit=False,
)
# Load PDA adapter (we'll switch between adapters)
model = PeftModel.from_pretrained(model, "adapter-pda", adapter_name="pda")
model.load_adapter("adapter-cot", adapter_name="cot")
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

FastLanguageModel.for_inference(model)
print("Model loaded with both adapters.")

In [ ]:
# === BASELINE (no adapter) ===
print("=" * 60)
print("BASELINE (Qwen3-1.7B, no adapter)")
print("=" * 60)
model.disable_adapter_layers()
base_results = eval_all(model, tokenizer, "Base")

# === PDA ADAPTER ===
print("\n" + "=" * 60)
print("PDA-DISTILLED (3 workers + merge)")
print("=" * 60)
model.enable_adapter_layers()
model.set_adapter("pda")
pda_results = eval_all(model, tokenizer, "PDA")

# === COT ADAPTER ===
print("\n" + "=" * 60)
print("COT-DISTILLED (single pass think-step-by-step)")
print("=" * 60)
model.set_adapter("cot")
cot_results = eval_all(model, tokenizer, "CoT")

## 6. Results

In [ ]:
print("\n" + "=" * 75)
print("  SIM 5c: PDA vs. CoT DISTILLATION")
print("=" * 75)
print(f"  Model: Qwen3-1.7B | QLoRA r=16, 3 epochs")
print(f"  PDA training: {len(pda_data)} examples | CoT training: {len(cot_data)} examples")
print("=" * 75)
print(f"\n  {'Benchmark':<12} {'Base':>8} {'CoT':>8} {'PDA':>8} {'PDA-CoT':>10}")
print("  " + "-" * 50)

pda_wins = 0
cot_wins = 0
for name in ["GSM8K", "MATH", "ARC-C"]:
    b = base_results[name]["accuracy"]
    c = cot_results[name]["accuracy"]
    p = pda_results[name]["accuracy"]
    diff = p - c
    sign = "+" if diff >= 0 else ""
    marker = " <--" if abs(diff) >= 3 else ""
    if diff > 0: pda_wins += 1
    elif diff < 0: cot_wins += 1
    print(f"  {name:<12} {b:>6.1f}%  {c:>6.1f}%  {p:>6.1f}%  {sign}{diff:>8.1f}pp{marker}")

print("  " + "-" * 50)
avg_b = sum(base_results[n]["accuracy"] for n in base_results) / 3
avg_c = sum(cot_results[n]["accuracy"] for n in cot_results) / 3
avg_p = sum(pda_results[n]["accuracy"] for n in pda_results) / 3
avg_diff = avg_p - avg_c
print(f"  {'Average':<12} {avg_b:>6.1f}%  {avg_c:>6.1f}%  {avg_p:>6.1f}%  +{avg_diff:>7.1f}pp")
print()

# Verdict
if avg_diff > 3:
    print("  VERDICT: PDA distillation significantly outperforms CoT distillation.")
    print("  Multi-perspective reasoning provides value beyond standard chain-of-thought.")
elif avg_diff > 0:
    print("  VERDICT: PDA distillation slightly outperforms CoT distillation.")
    print("  The multi-perspective approach may help, but the effect is small.")
elif avg_diff > -3:
    print("  VERDICT: PDA and CoT distillation perform similarly.")
    print("  The gains likely come from reasoning distillation in general, not multi-perspective specifically.")
else:
    print("  VERDICT: CoT distillation outperforms PDA distillation.")
    print("  Simpler reasoning may be easier for a small model to learn.")

# Save
results = {
    "base": base_results, "pda": pda_results, "cot": cot_results,
    "pda_examples": len(pda_data), "cot_examples": len(cot_data),
    "model": "Qwen3-1.7B", "method": "QLoRA r=16, 3ep",
}
with open("sim5c_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n  Saved to sim5c_results.json")